# Telecom Customer Churn Prediction — End-to-End Data Science Project
Business goal: predict which telecom customers are likely to leave (churn), so the retention team can contact high-risk customers.

Pipeline: CSV → Pandas/NumPy → EDA/Matplotlib → Feature Engineering → ML → Model Evaluation → PySpark → AWS.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay

df = pd.read_csv('telecom_customer_churn.csv')
df.head()

In [ ]:
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df['churn'].value_counts())

In [ ]:
# Basic numerical summary
df.describe(include='all').T

In [ ]:
df['churn'].value_counts().plot(kind='bar')
plt.title('Customer Churn Distribution')
plt.xlabel('Churn')
plt.ylabel('Number of Customers')
plt.show()

In [ ]:
df.groupby('contract_type')['churn'].apply(lambda x: (x == 'Yes').mean()).sort_values().plot(kind='bar')
plt.title('Churn Rate by Contract Type')
plt.ylabel('Churn Rate')
plt.show()

In [ ]:
# Feature engineering
df['avg_monthly_value'] = df['total_charges'] / df['tenure_months'].replace(0, np.nan)
df['support_call_rate'] = df['support_calls'] / df['tenure_months'].replace(0, np.nan)
df['payment_risk'] = df['late_payments'] / df['tenure_months'].replace(0, np.nan)
df['is_month_to_month'] = (df['contract_type'] == 'Month-to-Month').astype(int)
df['is_low_satisfaction'] = (df['satisfaction_score'] <= 2).astype(int)
df['is_high_support'] = (df['support_calls'] >= 4).astype(int)
df[['avg_monthly_value','support_call_rate','payment_risk','is_month_to_month','is_low_satisfaction','is_high_support']].head()

In [ ]:
# Separate features and target
X = df.drop(columns=['customer_id', 'churn'])
y = df['churn'].map({'No': 0, 'Yes': 1})

numeric_features = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])


In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]


In [ ]:
print('Accuracy :', round(accuracy_score(y_test, y_pred), 3))
print('Precision:', round(precision_score(y_test, y_pred), 3))
print('Recall   :', round(recall_score(y_test, y_pred), 3))
print('F1 Score :', round(f1_score(y_test, y_pred), 3))
print('ROC-AUC  :', round(roc_auc_score(y_test, y_prob), 3))
print('\nClassification Report:\n', classification_report(y_test, y_pred))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title('ROC Curve — Churn Model')
plt.show()

In [ ]:
# Predict churn risk for new customers
new_customer = pd.DataFrame([{
    'age': 29,
    'tenure_months': 6,
    'monthly_charges': 125,
    'total_charges': 750,
    'contract_type': 'Month-to-Month',
    'payment_method': 'UPI',
    'internet_service': 'Fiber',
    'support_calls': 5,
    'late_payments': 2,
    'data_usage_gb': 70,
    'satisfaction_score': 2,
    'avg_monthly_value': 125,
    'support_call_rate': 5/6,
    'payment_risk': 2/6,
    'is_month_to_month': 1,
    'is_low_satisfaction': 1,
    'is_high_support': 1
}])
risk = model.predict_proba(new_customer)[0, 1]
print(f'Predicted churn probability: {risk:.2%}')
print('Risk level:', 'HIGH' if risk >= 0.60 else 'MEDIUM' if risk >= 0.35 else 'LOW')

## PySpark version (for large datasets)
Use this section when the CSV is too large for a single-machine Pandas workflow.

In [ ]:
# Install once if needed: pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = SparkSession.builder.appName('TelecomChurn').getOrCreate()
spark_df = spark.read.csv('telecom_customer_churn.csv', header=True, inferSchema=True)
spark_df.printSchema()
spark_df.show(5)
spark_df.groupBy('contract_type').count().show()
spark_df.groupBy('churn').count().show()


## AWS deployment idea
1. Upload the CSV to Amazon S3.
2. Run the notebook/model on EC2 or a managed notebook environment.
3. Store the trained model artifact securely.
4. For a production system, expose predictions through an API and monitor model performance.
5. Never put AWS access keys directly inside the notebook.

## Interview explanation
I built a telecom customer churn prediction system. I loaded customer data using Pandas, cleaned missing values, performed EDA with Matplotlib, created business features such as payment risk and support-call rate, encoded categorical variables, trained a Logistic Regression classifier, and evaluated it using precision, recall, F1 and ROC-AUC. For scalable processing I used PySpark, and for cloud deployment I planned an S3/EC2 architecture on AWS.